# I9. Time-domain basics: pulsars and transients

The last notebook of the introductory track ({doc}`../index`). Imaging
(I7) adds up hours of data into one picture of a sky that is assumed not to
change. Pulsars and fast transients change in milliseconds. This notebook
simulates a MeerKAT L-band filterbank with a dispersed pulsar in it, in
NumPy, and finds the pulsar again. It downloads nothing and runs in under a
minute.

rang has no time-domain functions yet. Task V1 of ra-ng's vision
({gh}`research/vision.md`, section 3) names the first product, its data and
its oracle; this notebook switches to rang's functions when they exist.

**Learning goals.** After this notebook you can

- place pulsars, fast radio bursts (FRBs) and gamma-ray-burst (GRB)
  afterglows by time scale;
- derive and apply the dispersion delay Δt = k_DM · DM · (ν_lo⁻² − ν_hi⁻²);
- dedisperse a filterbank by shifting channels, and find the dispersion
  measure (DM) by trial;
- fold a periodic signal at its period;
- say what scattering does, and why an image averages a fast transient away.

In [ ]:
%matplotlib inline
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import constants as k

from rang import simtruth as st

rng = np.random.default_rng(20261111)

## 1. Time scales

| Class | Time scale | How it is found |
| --- | --- | --- |
| Pulsars | periodic pulses, periods of milliseconds to seconds, pulses a few percent of the period | periodicity search and folding of a beamformed or per-integration time series |
| FRBs | single bursts of about a millisecond, dispersed by DMs far above the Galaxy's | single-pulse search of dedispersed time series |
| GRB afterglows | slowly fading emission over days to months | repeated images: light curves of a position |

The first two need time resolution far below an imager's integration; the
third is a sequence of images. This notebook covers the first two.

## 2. Dispersion

Free electrons along the line of sight slow lower frequencies. A pulse
emitted at once arrives at frequency ν later than at infinite frequency by
k_DM · DM / ν², where DM = ∫ n_e dl is the **dispersion measure** in
pc cm⁻³ and k_DM = e² / (8π² ε₀ m_e c). From the constants:

In [ ]:
e, eps0, m_e, c = k.e.si.value, k.eps0.value, k.m_e.value, k.c.value
PC_CM3 = k.pc.value * 1e6  # 1 pc cm^-3 in m^-2
k_dm = e**2 / (8 * math.pi**2 * eps0 * m_e * c) * PC_CM3  # s Hz^2 per (pc cm^-3)
print(f"k_DM = {k_dm / 1e18 * 1e3:.4f} ms GHz^2 per pc cm^-3")


def delay(dm, f_hz, f_ref_hz):
    """Arrival delay (s) at f_hz relative to f_ref_hz."""
    return k_dm * dm * (f_hz**-2.0 - f_ref_hz**-2.0)


d = st.design("small")
print(f"MeerKAT L band, {d.fmin / 1e6:.0f}-{d.fmax / 1e6:.0f} MHz: DM 100 sweeps in {delay(100, d.fmin, d.fmax) * 1e3:.0f} ms")

Pulsar astronomers quote k_DM = 4.149 ms GHz² per pc cm⁻³ (the measured
constants give the same to four digits).

## 3. A simulated filterbank

A **filterbank** is total power (Stokes I) against time and frequency, as a
beamformer or a single dish records it. Here: 512 channels across L band,
0.5 ms samples, 10 s, unit-variance noise, and a pulsar of period 0.3141 s,
DM 100 pc cm⁻³ and a 4 ms Gaussian pulse, too faint to see in a single
channel or pulse:

In [ ]:
nchan, tsamp, duration = 512, 0.5e-3, 10.0
freq = np.linspace(d.fmax, d.fmin, nchan, endpoint=False) - (d.fmax - d.fmin) / nchan / 2  # high to low, as filterbanks are stored
nsamp = int(duration / tsamp)
t = np.arange(nsamp) * tsamp
period, dm_true, width, amplitude = 0.3141, 100.0, 4e-3, 0.3

# time from the nearest pulse centre (s); each channel is delayed by its dispersion
centre = 0.4  # pulse phase of the pulse centre, inside a fold bin (64 bins)
dist = np.abs(((t[None, :] - delay(dm_true, freq, freq[0])[:, None] - (centre - 0.5) * period) % period) - period / 2)
signal = amplitude * np.exp(-0.5 * (dist / (width / 2.355)) ** 2)
fb = (rng.standard_normal((nchan, nsamp)) + signal).astype(np.float32)
print(f"filterbank {fb.shape}, {fb.nbytes / 1e6:.0f} MB; peak signal per sample and channel {amplitude} sigma")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].imshow(fb[:, :2000], aspect="auto", cmap="Greys", extent=[0, 2000 * tsamp, freq[-1] / 1e6, freq[0] / 1e6])
axes[0].set(xlabel="time (s)", ylabel="frequency (MHz)", title="the data: noise")
axes[1].imshow(signal[:, :2000], aspect="auto", cmap="Greys", extent=[0, 2000 * tsamp, freq[-1] / 1e6, freq[0] / 1e6])
axes[1].set(xlabel="time (s)", title="the signal in it: dispersed pulses")
plt.show()

## 4. Dedispersion

Shift each channel earlier by its delay and sum over frequency. At the right
DM the pulses line up; at a wrong one they smear. (Real dedispersion codes
use faster tree or FFT algorithms; the principle is this.)

In [ ]:
def dedisperse(fb, dm):
    shifts = np.round(delay(dm, freq, freq[0]) / tsamp).astype(int)
    out = np.zeros(fb.shape[1])
    for c in range(fb.shape[0]):
        out += np.roll(fb[c], -shifts[c])
    return out / math.sqrt(fb.shape[0])  # unit variance for noise


def fold(series, p, nbins=64):
    bins = ((np.arange(len(series)) * tsamp / p) % 1 * nbins).astype(int)
    total = np.bincount(bins, series, nbins)
    count = np.bincount(bins, minlength=nbins)
    return total / np.sqrt(count)  # unit variance per bin for noise


def snr(profile):
    off = np.sort(profile)[: len(profile) // 2]  # the lower half as the off-pulse estimate
    return (profile.max() - np.median(profile)) / off.std()


fig, axes = plt.subplots(1, 3, figsize=(13, 3.3))
for ax, dm in zip(axes, (0.0, 50.0, dm_true)):
    ax.plot(np.linspace(0, 1, 64, endpoint=False), fold(dedisperse(fb, dm), period))
    ax.set(title=f"folded at the period, DM {dm:.0f}", xlabel="pulse phase")
axes[0].set_ylabel("S/N per bin")
plt.show()

## 5. Searching DM

With the DM unknown, try many and keep the best. The S/N of the folded
profile against trial DM:

In [ ]:
trials = np.arange(0, 201, 2.0)
curve = np.array([snr(fold(dedisperse(fb, dm), period)) for dm in trials])
best = trials[np.argmax(curve)]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(trials, curve, ".-")
ax.axvline(dm_true, color="k", lw=0.8)
ax.set(xlabel="trial DM (pc cm^-3)", ylabel="S/N of the folded profile", title=f"best trial DM {best:.0f}, true {dm_true:.0f}")
plt.show()

The curve's width is set by the pulse width: a DM error δ smears the pulse by
k_DM δ (ν_lo⁻² − ν_hi⁻²), which must stay below the width:

In [ ]:
print(f"DM error that smears the pulse by its own width ({width * 1e3:.0f} ms): {width / delay(1.0, freq[-1], freq[0]):.1f} pc cm^-3")
dnu = freq[0] - freq[1]
smear = 2 * k_dm * dm_true * dnu / freq[-1] ** 3
print(f"smearing within one {dnu / 1e6:.2f} MHz channel at {freq[-1] / 1e6:.0f} MHz, DM {dm_true:.0f}: {smear * 1e3:.2f} ms")

Intra-channel smearing is why pulsar data need narrow channels: it grows
as DM Δν / ν³ and cannot be removed by shifting whole channels.

## 6. Folding needs the right period

A small period error moves each pulse a little later than the last; over N
pulses the drift adds up. Fold at the true period and at periods off by one
part in 10⁴ and in 10³:

In [ ]:
series = dedisperse(fb, dm_true)
npulse = duration / period
fig, ax = plt.subplots(figsize=(7, 3))
for frac in (0.0, 1e-4, 1e-3):
    ax.plot(np.linspace(0, 1, 64, endpoint=False), fold(series, period * (1 + frac)), label=f"period × (1 + {frac:g}): drift {frac * npulse * period * 1e3:.1f} ms")
ax.set(xlabel="pulse phase", ylabel="S/N per bin", title=f"{npulse:.0f} pulses folded")
ax.legend(fontsize=8)
plt.show()

## 7. Scattering

Multipath propagation through the ionised interstellar medium delays part
of the signal, giving the pulse an exponential tail whose time constant
grows steeply towards low frequency, roughly as ν^−4. With an example
τ = 2 ms at 1 GHz:

In [ ]:
tt = np.arange(-20e-3, 60e-3, tsamp)
pulse = np.exp(-0.5 * (tt / (width / 2.355)) ** 2)
fig, ax = plt.subplots(figsize=(7, 3))
for f in (0.9e9, 1.3e9, 1.7e9):
    tau = 2e-3 * (f / 1e9) ** -4.0
    kernel = np.where(tt >= 0, np.exp(-np.clip(tt, 0, None) / tau), 0)
    scattered = np.convolve(pulse, kernel, mode="same")
    ax.plot(tt * 1e3, scattered / scattered.max(), label=f"{f / 1e9:.1f} GHz, tau {tau * 1e3:.1f} ms")
ax.set(xlabel="time (ms)", ylabel="normalised power", title="a 4 ms pulse after scattering")
ax.legend()
plt.show()

Unlike dispersion, scattering cannot be undone by shifting channels; it
lowers the peak S/N of a search.

## 8. Why an image averages a transient away

A burst of flux S lasting t_b inside an integration of length t_int appears
in that integration's visibilities with mean flux S t_b / t_int. For a
1 Jy, 1 ms burst seen by 64 MeerKAT antennas across the band (SEFD 420 Jy,
I1):

In [ ]:
nbl = 64 * 63 // 2
band = d.fmax - d.fmin
for t_int in (8.0, 2.0, 0.01):
    seen = 1.0 * 1e-3 / t_int
    noise = d.sefd / math.sqrt(2 * band * t_int) / math.sqrt(2) / math.sqrt(nbl)  # Stokes I snapshot image
    print(f"{t_int:>5} s images: burst appears as {seen * 1e3:7.3f} mJy, snapshot noise {noise * 1e3:.3f} mJy/beam, S/N {seen / noise:.1f}")

The burst's S/N in an image grows as 1/√t_int when integrations shorten
towards the burst's length: transient searches with an interferometer use
per-integration images or a beamformed time series, and image-plane light
curves and dynamic spectra (flux against time and frequency at one
position) recover what a deep image averages away. These are among the
candidate products for V1.

## Exercises

1. Add a single FRB-like burst (one pulse, DM 600, width 1 ms) to the
   filterbank. Find it with a single-pulse search: the maximum of the
   dedispersed series against trial DM.
2. Halve the number of channels. How does the intra-channel smearing change,
   and the S/N at the true DM?
3. Search period as well as DM: fold at periods within ±10⁻³ of the true one
   and map S/N over (DM, period).
4. With scattering τ = 20 ms at 1 GHz, which part of the band would you keep
   for a search, and what does that cost in S/N?

## Where next

- {gh}`research/vision.md`, section 3: the time-series scope of ra-ng.
- {doc}`i1-radio-sky` for the radiometer equation used in section 8.
- {doc}`i7-imaging-and-deconvolution`: what an image does with the data.